<a href="https://colab.research.google.com/github/ridamumtazz/Flyrank-ML-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

## 1. Ranked Actions + Reason Codes

The content queue ranks content items by their measured action score and assigns a recommended action with a reason code. The purpose is to help a content or SEO team decide which items should be reviewed first.

### Action 1 — Improve

**When to use:** Content with strong observed opportunity for improvement, such as high impressions but weaker clicks, declining performance, or other measured signals suggesting that the content may benefit from optimization.

**Reason codes:**

* `RANKING_OPPORTUNITY` — The content receives search visibility but may have an opportunity to improve its search performance.
* `DECLINING_PERFORMANCE` — Observed performance is moving downward and may require investigation.
* `LOW_ENGAGEMENT` — Observed engagement is relatively weak and may indicate that the content needs improvement.

### Action 2 — Review

**When to use:** Content where the signals are mixed or where the model cannot provide a strong enough recommendation for immediate improvement.

**Reason codes:**

* `MIXED_SIGNALS` — Different performance signals point in different directions.
* `NEEDS_HUMAN_REVIEW` — The item requires additional context before deciding on an action.
* `UNCERTAIN_PRIORITY` — The measured signals do not provide enough evidence for a strong recommendation.

### Action 3 — Monitor

**When to use:** Content that currently does not show a strong measured opportunity or urgent decline.

**Reason codes:**

* `STABLE_PERFORMANCE` — Observed performance is relatively stable.
* `LOW_PRIORITY` — The measured opportunity is currently limited.
* `NO_CLEAR_ACTION` — Available signals do not indicate a clear improvement action.

### Priority Order

The recommended priority order is:

1. **Improve** — investigate first when the measured opportunity is strong.
2. **Review** — investigate when signals are mixed or uncertain.
3. **Monitor** — continue observing unless new evidence changes the priority.

These recommendations are directional decision-support. A high-ranked item should be reviewed by a human before any content changes are made.


In [4]:
import os

print("Current directory:")
print(os.getcwd())

print("\nFiles and folders:")
for root, dirs, files in os.walk(".", topdown=True):
    # Skip unnecessary system folders
    dirs[:] = [d for d in dirs if d not in [".git", "__pycache__"]]

    for file in files:
        print(os.path.join(root, file))


Current directory:
/content

Files and folders:
./.config/.last_opt_in_prompt.yaml
./.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
./.config/config_sentinel
./.config/active_config
./.config/.last_survey_prompt.yaml
./.config/.last_update_check.json
./.config/default_configs.db
./.config/gce
./.config/configurations/config_default
./.config/logs/2026.06.04/13.32.38.346437.log
./.config/logs/2026.06.04/13.32.21.210570.log
./.config/logs/2026.06.04/13.31.42.499627.log
./.config/logs/2026.06.04/13.32.39.344962.log
./.config/logs/2026.06.04/13.32.18.735754.log
./.config/logs/2026.06.04/13.32.02.654775.log
./sample_data/README.md
./sample_data/anscombe.json
./sample_data/mnist_train_small.csv
./sample_data/california_housing_train.csv
./sample_data/california_housing_test.csv
./sample_data/mnist_test.csv


In [6]:
from google.colab import userdata
from huggingface_hub import hf_hub_download
import pandas as pd
import numpy as np
import os

# Get Hugging Face token
HF_TOKEN = userdata.get("HF_TOKEN")

# Download FlyRank sample dataset
sample_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance_sample.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

# Load dataset
sample_df = pd.read_parquet(sample_path)

print("Dataset loaded successfully!")
print("Rows:", sample_df.shape[0])
print("Columns:", sample_df.shape[1])

fact_content_daily_performance_sample.pa(…): reconstructing file:   0%|          |  0.00B /  145MB            

fact_content_daily_performance_sample.pa(…): downloading bytes:           |  0.00B            

Dataset loaded successfully!
Rows: 11694072
Columns: 31


In [10]:
# ============================================
# ML-10 SECTION 1 - CREATE ACTION QUEUE
# ============================================

import pandas as pd
import numpy as np
import os

# Make a copy
df = sample_df.copy()

# Convert date
df["report_date"] = pd.to_datetime(
    df["report_date"]
)

# Create CTR
df["ctr"] = np.where(
    df["gsc_impressions"] > 0,
    df["gsc_clicks"] / df["gsc_impressions"],
    0
)

# Create engagement rate
df["engagement_rate"] = np.where(
    df["ga4_sessions"] > 0,
    df["ga4_engaged_sessions"] / df["ga4_sessions"],
    0
)

# Sort by content and date
df = df.sort_values(
    ["content_hash_id", "report_date"]
)

# Keep the latest observation for each content item
latest_df = (
    df
    .groupby("content_hash_id")
    .tail(1)
    .copy()
)

print("Latest content records:", len(latest_df))

Latest content records: 409205


In [11]:
# ============================================
# ML-10 SECTION 1 - ACTION SCORE + REASON CODES
# ============================================

# Create percentile-based scores
latest_df["impression_score"] = (
    latest_df["gsc_impressions"]
    .rank(pct=True)
)

latest_df["click_score"] = (
    latest_df["gsc_clicks"]
    .rank(pct=True)
)

latest_df["session_score"] = (
    latest_df["ga4_sessions"]
    .rank(pct=True)
)

latest_df["position_score"] = (
    1 - (
        latest_df["gsc_avg_position"]
        .rank(pct=True)
    )
)

# Handle missing / invalid average position
latest_df.loc[
    latest_df["gsc_avg_position"] <= 0,
    "position_score"
] = 0

# Create a simple directional action score
latest_df["action_score"] = (
    0.40 * latest_df["impression_score"]
    + 0.30 * (1 - latest_df["click_score"])
    + 0.20 * (1 - latest_df["session_score"])
    + 0.10 * latest_df["position_score"]
)

# Convert to 0-100 scale
latest_df["action_score"] = (
    latest_df["action_score"] * 100
)

# Assign action categories
latest_df["action"] = np.select(
    [
        latest_df["action_score"] >= 70,
        latest_df["action_score"] >= 40
    ],
    [
        "Improve",
        "Review"
    ],
    default="Monitor"
)

# Create reason codes
latest_df["reason_code"] = np.select(
    [
        (
            (latest_df["gsc_impressions"] > 0) &
            (latest_df["gsc_clicks"] == 0)
        ),
        (
            (latest_df["gsc_impressions"] > 0) &
            (latest_df["ctr"] < latest_df["ctr"].median())
        ),
        (
            latest_df["gsc_avg_position"] > 10
        )
    ],
    [
        "RANKING_OPPORTUNITY",
        "LOW_CTR",
        "LOW_RANKING"
    ],
    default="MIXED_SIGNALS"
)

# Sort by action score
latest_df = latest_df.sort_values(
    "action_score",
    ascending=False
).reset_index(drop=True)

# Add rank
latest_df["rank"] = (
    latest_df.index + 1
)

print("Action queue created.")
print("Total items:", len(latest_df))

print("\nAction distribution:")
print(
    latest_df["action"].value_counts()
)

print("\nReason code distribution:")
print(
    latest_df["reason_code"].value_counts()
)

print("\nTop 20 ranked items:")
display(
    latest_df[
        [
            "rank",
            "content_hash_id",
            "action_score",
            "action",
            "reason_code",
            "gsc_impressions",
            "gsc_clicks",
            "ctr",
            "gsc_avg_position"
        ]
    ].head(20)
)

Action queue created.
Total items: 409205

Action distribution:
action
Monitor    306795
Review      94833
Improve      7577
Name: count, dtype: int64

Reason code distribution:
reason_code
MIXED_SIGNALS          288066
RANKING_OPPORTUNITY    117794
LOW_RANKING              3345
Name: count, dtype: int64

Top 20 ranked items:


,rank,content_hash_id,action_score,action,reason_code,gsc_impressions,gsc_clicks,ctr,gsc_avg_position
0,1,content_69620c40b8f0db42,75.326558,Improve,RANKING_OPPORTUNITY,1089,0,0.0,0.090909
1,2,content_d835bf58b02a8f91,75.197891,Improve,RANKING_OPPORTUNITY,562,0,0.0,0.092527
2,3,content_9a56a1eb8a8ca845,74.948512,Improve,RANKING_OPPORTUNITY,814,0,0.0,1.171990
3,4,content_8e196725f179b532,74.840235,Improve,RANKING_OPPORTUNITY,1028,0,0.0,1.560311
4,5,content_a33031274394cf93,74.805477,Improve,RANKING_OPPORTUNITY,921,0,0.0,1.636265
5,6,content_c567f95274159a02,74.740165,Improve,RANKING_OPPORTUNITY,998,0,0.0,1.884770
6,7,content_60b9f1e68c286162,74.688597,Improve,RANKING_OPPORTUNITY,498,0,0.0,1.508032
7,8,content_3b909504f09603f0,74.669342,Improve,RANKING_OPPORTUNITY,594,0,0.0,1.762626
8,9,content_46235b88039a0566,74.623256,Improve,RANKING_OPPORTUNITY,350,0,0.0,1.362857
9,10,content_b98fd317f3fa7210,74.583433,Improve,RANKING_OPPORTUNITY,399,0,0.0,1.604010


### Queue Interpretation

The regenerated action queue contains 409,205 unique content items based on their latest available observations in the warehouse sample. The queue assigns 7,577 items to Improve, 94,833 to Review, and 306,795 to Monitor.

The highest-ranked items are mainly marked as `RANKING_OPPORTUNITY`. These items have observed search impressions but zero recorded clicks in the latest observation, making them candidates for human investigation. The queue is intended to help prioritize review rather than automatically determine that these pages need changes.

The reason codes provide a simple explanation for why an item appears in the queue. `RANKING_OPPORTUNITY` highlights items with observed impressions but no recorded clicks, while `LOW_RANKING` identifies items with weaker observed average positions. Items that do not meet these conditions receive `MIXED_SIGNALS` and should be reviewed more carefully before action.

This regenerated queue is based on the warehouse sample and latest available observations. It is therefore treated as a reproducible ML-10 action queue rather than an exact recreation of the original 30,000-row ML-07 queue.


### Archetype → Action Mapping

| Content archetype              | Observed signal                                          | Recommended action |
| ------------------------------ | -------------------------------------------------------- | ------------------ |
| High impressions, zero clicks  | Search visibility is observed but no clicks are recorded | Improve            |
| Lower ranking                  | Average position is weaker than the selected threshold   | Review or Improve  |
| Mixed signals                  | No single strong opportunity is identified               | Review             |
| Stable or low-priority signals | No clear measured opportunity for immediate action       | Monitor            |

The archetype mapping is a prioritization guide rather than a fixed rule. A human reviewer should confirm the reason behind the observed pattern before making content changes.


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## 2. Intended Use and Limits

### Intended Use

The content action playbook is designed to help SEO and content teams prioritize which content items should be investigated first. The ranked queue uses measured performance signals such as impressions, clicks, CTR, average position, sessions, and engagement to provide a directional recommendation.

The main purpose is to reduce the amount of manual work needed to review a large content library. Instead of checking every content item with the same priority, a team can start with items ranked as `Improve`, then review items marked as `Review`, while continuing to monitor lower-priority items.

The playbook is intended for content strategists, SEO specialists, editors, and marketing teams. It provides decision-support and helps organize human review.

### Limits

The recommendations are based on observed historical performance and should not be treated as guaranteed predictions of future results. A high action score does not prove that changing a page will increase clicks, traffic, or conversions.

The ML-09 validation also showed that model performance changed when moving from a time-aware split to a grouped-by-client split. This indicates that generalization to unseen clients is weaker and that the results should be interpreted carefully.

The current action queue is also based on the latest available observation for each content item in the warehouse sample. It should be refreshed when new data becomes available because content performance can change over time.

The playbook should not be used as a fully automated production system. It is a prioritization tool that supports human decision-making.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## 3. Human Review + The No-Go List

### Human Review Before Action

Every high-priority recommendation should be reviewed by a person before any content action is taken.

The reviewer should check:

1. **Content quality:** Is the page accurate, useful, and up to date?
2. **Search intent:** Does the content match what users are searching for?
3. **Business value:** Is the content important to the organization's goals?
4. **Recent changes:** Was the page recently updated or changed?
5. **Seasonality:** Could the observed performance be affected by seasonal demand?
6. **Technical factors:** Are indexing, crawling, links, or technical issues affecting performance?
7. **External factors:** Could competitors or search engine changes explain the observed performance?

A high action score should be treated as a starting point for investigation rather than a final decision.

### No-Go List

The following actions should never be fully automated based only on this model:

* Automatically publishing or editing content.
* Automatically deleting or redirecting pages.
* Automatically changing legal, medical, financial, or safety-related information.
* Automatically making decisions that affect people without human review.
* Automatically changing content only because it has a low score.
* Treating the action score as proof that a content change will improve performance.
* Automatically replacing expert judgment with a model recommendation.

The model should prioritize content for review, while final decisions remain with qualified human reviewers.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [ ]:
## 4. Monitoring / Retrain Triggers

The content action recommendations may become stale as search behavior, rankings, competition, and user behavior change. The queue should therefore be monitored and refreshed regularly.

### Monitoring Triggers

The system should be reviewed when:

* The distribution of `Improve`, `Review`, and `Monitor` recommendations changes substantially.
* The number of high-priority recommendations increases unexpectedly.
* Recent prediction errors become larger than the measured validation results.
* Feature distributions change noticeably compared with the data used during model development.
* Search behavior or traffic patterns change significantly.
* Major search engine updates affect content performance.
* New content types or client groups appear that were not well represented in the training data.

### Retraining Triggers

Retraining should be considered when measured prediction errors increase consistently on newer data, when data distributions change substantially, or when the relationship between performance signals and future outcomes appears to change.

The action queue should also be refreshed when new performance observations become available. Recommendations should not be treated as permanent because content performance can decay or change over time.

A single unusual prediction should not automatically trigger retraining. Retraining decisions should be based on broader evidence across multiple observations.

### Decay and Refresh

Content recommendations can lose relevance as time passes. A page that is a high-priority improvement opportunity today may perform differently later because of ranking changes, seasonality, competition, or changes in search demand.

For this reason, the queue should be regenerated periodically using recent observations. Previous recommendations should be re-evaluated rather than assumed to remain valid indefinitely.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## 5. Exports for the Paper

The final ranked action queue is exported to `work/outputs/` so it can be reused in the research paper and future analysis. The exported queue contains the rank, action score, recommended action, reason code, and supporting performance signals.

The queue contains 409,205 content items generated from the latest available observation for each content item in the warehouse sample. It is intended as a reproducible decision-support output rather than a permanent list of recommendations.

The exported file is:

`work/outputs/content_action_playbook_queue.csv`

The queue can be regenerated when new data becomes available, allowing recommendations to be refreshed as content performance changes over time.


In [13]:
# ============================================
# ML-10 SECTION 5 - EXPORT QUEUE FOR PAPER
# ============================================

import os

# Create output directory
os.makedirs("work/outputs", exist_ok=True)

# Select columns for export
export_columns = [
    "rank",
    "content_hash_id",
    "action_score",
    "action",
    "reason_code",
    "gsc_impressions",
    "gsc_clicks",
    "ctr",
    "gsc_avg_position",
    "ga4_sessions",
    "engagement_rate",
    "report_date"
]

# Keep only columns that exist
export_columns = [
    col for col in export_columns
    if col in latest_df.columns
]

# Create final queue
final_queue = latest_df[export_columns].copy()

# Export CSV
output_path = "work/outputs/content_action_playbook_queue.csv"

final_queue.to_csv(
    output_path,
    index=False
)

print("================================")
print("EXPORT COMPLETE")
print("================================")
print("File:", output_path)
print("Rows:", len(final_queue))
print("Columns:", len(final_queue.columns))

print("\nTop 10 exported recommendations:")
display(final_queue.head(10))

EXPORT COMPLETE
File: work/outputs/content_action_playbook_queue.csv
Rows: 409205
Columns: 12

Top 10 exported recommendations:


,rank,content_hash_id,action_score,action,reason_code,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,ga4_sessions,engagement_rate,report_date
0,1,content_69620c40b8f0db42,75.326558,Improve,RANKING_OPPORTUNITY,1089,0,0.0,0.090909,0.0,0.0,2026-06-30
1,2,content_d835bf58b02a8f91,75.197891,Improve,RANKING_OPPORTUNITY,562,0,0.0,0.092527,0.0,0.0,2026-06-30
2,3,content_9a56a1eb8a8ca845,74.948512,Improve,RANKING_OPPORTUNITY,814,0,0.0,1.171990,0.0,0.0,2026-06-30
3,4,content_8e196725f179b532,74.840235,Improve,RANKING_OPPORTUNITY,1028,0,0.0,1.560311,0.0,0.0,2026-06-30
4,5,content_a33031274394cf93,74.805477,Improve,RANKING_OPPORTUNITY,921,0,0.0,1.636265,0.0,0.0,2026-06-30
5,6,content_c567f95274159a02,74.740165,Improve,RANKING_OPPORTUNITY,998,0,0.0,1.884770,0.0,0.0,2026-06-30
6,7,content_60b9f1e68c286162,74.688597,Improve,RANKING_OPPORTUNITY,498,0,0.0,1.508032,0.0,0.0,2026-06-30
7,8,content_3b909504f09603f0,74.669342,Improve,RANKING_OPPORTUNITY,594,0,0.0,1.762626,0.0,0.0,2026-06-30
8,9,content_46235b88039a0566,74.623256,Improve,RANKING_OPPORTUNITY,350,0,0.0,1.362857,0.0,0.0,2026-06-30
9,10,content_b98fd317f3fa7210,74.583433,Improve,RANKING_OPPORTUNITY,399,0,0.0,1.604010,0.0,0.0,2026-06-30


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.